In [115]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [116]:
dataset = pd.read_csv('Weather_Data.csv')
dataset.head(10)

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2/1/2008,19.5,22.4,15.6,6.2,0.0,W,41,S,SSW,...,92,84,1017.6,1017.4,8,8,20.7,20.9,Yes,Yes
1,2/2/2008,19.5,25.6,6.0,3.4,2.7,W,41,W,E,...,83,73,1017.9,1016.4,7,7,22.4,24.8,Yes,Yes
2,2/3/2008,21.6,24.5,6.6,2.4,0.1,W,41,ESE,ESE,...,88,86,1016.7,1015.6,7,8,23.5,23.0,Yes,Yes
3,2/4/2008,20.2,22.8,18.8,2.2,0.0,W,41,NNE,E,...,83,90,1014.2,1011.8,8,8,21.4,20.9,Yes,Yes
4,2/5/2008,19.7,25.7,77.4,4.8,0.0,W,41,NNE,W,...,88,74,1008.3,1004.8,8,8,22.5,25.5,Yes,Yes
5,2/6/2008,20.2,27.2,1.6,2.6,8.6,W,41,W,ENE,...,69,62,1002.7,998.6,6,6,23.8,26.0,Yes,Yes
6,2/7/2008,18.6,26.3,6.2,5.2,5.2,W,41,W,S,...,75,80,999.0,1000.3,4,7,21.7,22.3,Yes,Yes
7,2/8/2008,17.2,22.3,27.6,5.8,2.1,W,41,S,SE,...,77,61,1008.3,1007.4,7,8,18.9,21.1,Yes,Yes
8,2/9/2008,16.4,20.8,12.6,4.8,3.0,W,41,SSW,W,...,92,91,1006.4,1007.6,7,7,17.1,16.5,Yes,Yes
9,2/10/2008,14.6,24.2,8.8,4.4,10.1,W,41,W,SSE,...,80,53,1014.0,1013.4,4,2,17.2,23.3,Yes,No


In [117]:
dataset['Month'] = pd.to_datetime(dataset['Date']).dt.month

In [118]:
dataset.head()

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,Month
0,2/1/2008,19.5,22.4,15.6,6.2,0.0,W,41,S,SSW,...,84,1017.6,1017.4,8,8,20.7,20.9,Yes,Yes,2
1,2/2/2008,19.5,25.6,6.0,3.4,2.7,W,41,W,E,...,73,1017.9,1016.4,7,7,22.4,24.8,Yes,Yes,2
2,2/3/2008,21.6,24.5,6.6,2.4,0.1,W,41,ESE,ESE,...,86,1016.7,1015.6,7,8,23.5,23.0,Yes,Yes,2
3,2/4/2008,20.2,22.8,18.8,2.2,0.0,W,41,NNE,E,...,90,1014.2,1011.8,8,8,21.4,20.9,Yes,Yes,2
4,2/5/2008,19.7,25.7,77.4,4.8,0.0,W,41,NNE,W,...,74,1008.3,1004.8,8,8,22.5,25.5,Yes,Yes,2


In [119]:
print(dataset.isnull().sum())

Date             0
MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustDir      0
WindGustSpeed    0
WindDir9am       0
WindDir3pm       0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
RainToday        0
RainTomorrow     0
Month            0
dtype: int64


In [120]:
dataset = dataset.dropna()

In [121]:
datatrain, dataset = train_test_split(dataset, test_size=0.2, random_state=42)

In [122]:
X_train1 = datatrain.drop(columns=['RainToday', 'RainTomorrow', 'Date', 'Temp9am', 'Humidity9am', 'Pressure9am', 'Rainfall'])
X_train2 = datatrain.drop(columns=['RainTomorrow', 'Date'])
y_train1 = datatrain['RainToday']
y_train2 = datatrain['RainTomorrow']

X_test1 = dataset.drop(columns=['RainToday', 'RainTomorrow', 'Date', 'Temp9am', 'Humidity9am', 'Pressure9am', 'Rainfall'])
X_test2 = dataset.drop(columns=['RainTomorrow', 'Date'])
y_test1 = dataset['RainToday']
y_test2 = dataset['RainTomorrow']

In [123]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cols = ['WindGustDir', 'WindDir9am', 'WindDir3pm']
for i in cols:
  X_train1[i] = le.fit_transform(X_train1[i])
  X_train2[i] = le.fit_transform(X_train2[i])
  X_test1[i] = le.transform(X_test1[i])
  X_test2[i] = le.transform(X_test2[i])
X_train2['RainToday'] = le.fit_transform(X_train2['RainToday'])
X_test2['RainToday'] = le.transform(X_test2['RainToday'])

In [124]:
map={'No': 0, 'Yes': 1}
y_train1 = y_train1.map(map)
y_train2 = y_train2.map(map)
y_test1 = y_test1.map(map)
y_test2 = y_test2.map(map)

In [125]:
from catboost import CatBoostClassifier
model1 = CatBoostClassifier()
model1.fit(X_train1, y_train1)
y_pred1 = model1.predict(X_test1)

Learning rate set to 0.015533
0:	learn: 0.6855965	total: 2.7ms	remaining: 2.7s
1:	learn: 0.6777319	total: 4.57ms	remaining: 2.28s
2:	learn: 0.6695836	total: 6.54ms	remaining: 2.17s
3:	learn: 0.6615746	total: 8.59ms	remaining: 2.14s
4:	learn: 0.6548854	total: 10.1ms	remaining: 2.01s
5:	learn: 0.6485293	total: 11.6ms	remaining: 1.93s
6:	learn: 0.6416487	total: 13.3ms	remaining: 1.88s
7:	learn: 0.6358015	total: 14.8ms	remaining: 1.84s
8:	learn: 0.6301244	total: 16.9ms	remaining: 1.86s
9:	learn: 0.6248915	total: 18.4ms	remaining: 1.82s
10:	learn: 0.6189454	total: 20ms	remaining: 1.8s
11:	learn: 0.6133985	total: 21.7ms	remaining: 1.79s
12:	learn: 0.6085666	total: 23.3ms	remaining: 1.77s
13:	learn: 0.6034314	total: 24.9ms	remaining: 1.75s
14:	learn: 0.5981725	total: 27ms	remaining: 1.77s
15:	learn: 0.5926684	total: 29.2ms	remaining: 1.79s
16:	learn: 0.5874872	total: 30.8ms	remaining: 1.78s
17:	learn: 0.5822308	total: 32.3ms	remaining: 1.76s
18:	learn: 0.5782470	total: 34.1ms	remaining: 1.76s

In [126]:
score1 = accuracy_score(y_test1, y_pred1)
score1

0.8122137404580153

In [127]:
from catboost import CatBoostClassifier
model2 = CatBoostClassifier(iterations=2000)
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

Learning rate set to 0.008227
0:	learn: 0.6874361	total: 2.06ms	remaining: 4.12s
1:	learn: 0.6811576	total: 4ms	remaining: 3.99s
2:	learn: 0.6753987	total: 6.02ms	remaining: 4.01s
3:	learn: 0.6694622	total: 8.19ms	remaining: 4.09s
4:	learn: 0.6633277	total: 10ms	remaining: 3.99s
5:	learn: 0.6579149	total: 11.9ms	remaining: 3.94s
6:	learn: 0.6531279	total: 13.8ms	remaining: 3.94s
7:	learn: 0.6475216	total: 15.6ms	remaining: 3.88s
8:	learn: 0.6428140	total: 17.2ms	remaining: 3.81s
9:	learn: 0.6378832	total: 19.4ms	remaining: 3.85s
10:	learn: 0.6330680	total: 21.3ms	remaining: 3.84s
11:	learn: 0.6286728	total: 22.9ms	remaining: 3.79s
12:	learn: 0.6238919	total: 24.5ms	remaining: 3.75s
13:	learn: 0.6188712	total: 26ms	remaining: 3.69s
14:	learn: 0.6135101	total: 27.4ms	remaining: 3.63s
15:	learn: 0.6092564	total: 28.9ms	remaining: 3.58s
16:	learn: 0.6044765	total: 30.3ms	remaining: 3.54s
17:	learn: 0.6001545	total: 31.9ms	remaining: 3.51s
18:	learn: 0.5952381	total: 33.7ms	remaining: 3.51s

In [128]:
score2 = accuracy_score(y_test2, y_pred2)
score2

0.8274809160305343